# CryptoSphere Analytics - API Data Collection Pipeline

## Overview & Learning Objectives

Welcome to the **CryptoSphere Analytics Data Collection Pipeline**! This notebook demonstrates a complete, production-ready data acquisition system for cryptocurrency market data.

### What You'll Build
- **Professional API Integration** with CoinMarketCap's cryptocurrency data
- **Robust Error Handling** for production-ready data pipelines
- **Dual Storage Strategy** (CSV + SQL Database) for data reliability
- **Data Validation & Quality Checks** to ensure data integrity
- **Scalable Architecture** ready for automation and cloud deployment

### Business Value
- **Real-time cryptocurrency data** for market analysis
- **Historical data tracking** for trend identification
- **Data lineage and audit trails** for compliance
- **Foundation for ML models** and predictive analytics

### Technical Architecture
```
CoinMarketCap API → JSON Response → pandas DataFrame → CSV Files + SQL Database
                                                    ↓
                               Data Quality Validation & Error Handling
```

### Learning Outcomes
By completing this notebook, you'll understand:
- **API Integration Best Practices** (authentication, error handling, rate limiting)
- **Data Pipeline Design** (Extract, Transform, Load patterns)
- **Database Operations** (stored procedures, data types, transactions)
- **Data Quality Management** (validation, monitoring, alerting)
- **Production Considerations** (logging, error recovery, scalability)

---

## Prerequisites
Before running this notebook, ensure you have:
- **Database Setup**: Run `03-sql-processing/00-schema-setup/01-setup_all_layers.sql`
- **API Access**: Valid CoinMarketCap API key in `.env` file
- **Environment**: All packages from `requirements.txt` installed
- **Permissions**: Database write access and file system permissions

---

## Notebook Roadmap
1. **Environment Setup** → Configure API access and dependencies
2. **Data Collection** → Fetch cryptocurrency data from CoinMarketCap API
3. **Data Processing** → Transform JSON into structured DataFrames
4. **Data Storage** → Save to CSV files and SQL database
5. **Quality Validation** → Verify data integrity and completeness
6. **Pipeline Testing** → End-to-end workflow validation

Let's build something amazing!

# PHASE 1: Environment Setup & Configuration

## Objective
Set up the complete environment for cryptocurrency data collection, including:
- Python library imports and dependency management
- API authentication and configuration
- Database connectivity setup
- Error handling and logging infrastructure

## Why This Matters
Proper environment setup is the foundation of any production data pipeline. We're establishing:
- **Reliability**: Robust error handling from the start
- **Security**: Safe credential management
- **Scalability**: Modular design for future enhancements
- **Maintainability**: Clear separation of concerns

---

## Library Imports: Setting Up Our Data Pipeline Toolkit

**What this cell does**: Import all the essential Python libraries needed for our cryptocurrency data collection pipeline.

In [49]:
# 1.1 Import Required Libraries

# Core data processing
import pandas as pd
import numpy as np

# API interactions
from requests import Request, Session
from requests.exceptions import ConnectionError, Timeout, TooManyRedirects
import json

# Date and file operations
from datetime import datetime
import time
import os

# Environment and database
from dotenv import load_dotenv
import pyodbc

print("All dependencies imported successfully!")
print("Libraries loaded:")
print("   • pandas, numpy - Data processing")
print("   • requests - API communication") 
print("   • pyodbc - Database connectivity")
print("   • dotenv - Environment management")

All dependencies imported successfully!
Libraries loaded:
   • pandas, numpy - Data processing
   • requests - API communication
   • pyodbc - Database connectivity
   • dotenv - Environment management


In [50]:
# 1.2 Load Environment Configuration

# Load environment variables from .env file
load_dotenv()

print("Environment variables loaded successfully!")
print("Tip: Make sure your .env file contains COINMARKETCAP_API_KEY")

Environment variables loaded successfully!
Tip: Make sure your .env file contains COINMARKETCAP_API_KEY


### API Authentication Setup

**What this cell does**: Load and validate CoinMarketCap API key from environment variables.
**Required**: Make sure your `.env` file contains `COINMARKETCAP_API_KEY=your_api_key_here`

In [51]:
# 1.3 Configure API Access

# Get API key from environment
COINMARKETCAP_API_KEY = os.getenv('COINMARKETCAP_API_KEY')

# Validate API key
if not COINMARKETCAP_API_KEY:
    print("ERROR: COINMARKETCAP_API_KEY not found!")
    print("Please add your API key to the .env file:")
    print("   COINMARKETCAP_API_KEY=your_api_key_here")
else:
    print("API key loaded successfully!")
    print(f"Key preview: {COINMARKETCAP_API_KEY[:4]}...")
    print("Ready to connect to CoinMarketCap API")

API key loaded successfully!
Key preview: 7957...
Ready to connect to CoinMarketCap API


## API Data Collection

**What this cell does**: Configure API parameters and fetch real-time cryptocurrency data from CoinMarketCap.
**Result**: Retrieves top 10 cryptocurrencies with market data (prices, volumes, market caps) in JSON format.

In [ ]:

# Target cryptocurrencies (Top 10 by market cap)
CRYPTO_SYMBOLS = ['BTC', 'ETH', 'BNB', 'USDT', 'XRP', 'SOL', 'USDC', 'DOGE', 'TRX', 'ADA']

url = 'https://pro-api.coinmarketcap.com/v1/cryptocurrency/listings/latest'
parameters = {
  'start':'1',
  'limit':'10',
  'convert':'USD'
}
headers = {
  'Accepts': 'application/json',
  'X-CMC_PRO_API_KEY': COINMARKETCAP_API_KEY,
}

# Create API session
session = Session()
session.headers.update(headers)

try:
  response = session.get(url, params=parameters)
  
  # Check if request was successful
  if response.status_code == 200:
      data = json.loads(response.text)
      # Pretty print the JSON response
      print(json.dumps(data, indent=2))
  else:
      print(f"Error: {response.status_code}")
      print(response.text)
  
except (ConnectionError, Timeout, TooManyRedirects) as e:
  print(e)


{
  "status": {
    "timestamp": "2025-10-12T09:52:41.622Z",
    "error_code": 0,
    "error_message": null,
    "elapsed": 11,
    "credit_count": 1,
    "notice": null,
    "total_count": 9517
  },
  "data": [
    {
      "id": 1,
      "name": "Bitcoin",
      "symbol": "BTC",
      "slug": "bitcoin",
      "num_market_pairs": 12415,
      "date_added": "2010-07-13T00:00:00.000Z",
      "tags": [
        "mineable",
        "pow",
        "sha-256",
        "store-of-value",
        "state-channel",
        "coinbase-ventures-portfolio",
        "three-arrows-capital-portfolio",
        "polychain-capital-portfolio",
        "binance-labs-portfolio",
        "blockchain-capital-portfolio",
        "boostvc-portfolio",
        "cms-holdings-portfolio",
        "dcg-portfolio",
        "dragonfly-capital-portfolio",
        "electric-capital-portfolio",
        "fabric-ventures-portfolio",
        "framework-ventures-portfolio",
        "galaxy-digital-portfolio",
        "huobi-capit

In [53]:
pd.set_option('display.max_columns', None)  # show all column

## JSON Data Processing

**What this cell does**: Extract and transform the raw JSON response into structured pandas DataFrames.
**Result**: Creates `df_crypto` (cryptocurrency market data) and `df_status` (API metadata) for analysis.

In [43]:
# Extract main sections
status_data = data['status']        # metadata about the API request
crypto_data = data['data']          # actual cryptocurrency list and info

# Normalize (flatten) both parts
df_crypto = pd.json_normalize(crypto_data, sep='_')
df_status = pd.json_normalize(status_data, sep='_')

# Display summaries
print("=== STATUS INFO ===")
print("Shape:", df_status.shape)
display(df_status)
print("\n=== CRYPTO DATA INFO ===")
print("Shape:", df_crypto.shape)
display(df_crypto)


=== STATUS INFO ===
Shape: (1, 7)


,timestamp,error_code,error_message,elapsed,credit_count,notice,total_count
0,2025-10-12T08:05:37.097Z,0,None,26,1,None,9517



=== CRYPTO DATA INFO ===
Shape: (10, 36)


,id,name,symbol,slug,num_market_pairs,date_added,tags,max_supply,circulating_supply,total_supply,infinite_supply,platform,cmc_rank,self_reported_circulating_supply,self_reported_market_cap,tvl_ratio,last_updated,quote_USD_price,quote_USD_volume_24h,quote_USD_volume_change_24h,quote_USD_percent_change_1h,quote_USD_percent_change_24h,quote_USD_percent_change_7d,quote_USD_percent_change_30d,quote_USD_percent_change_60d,quote_USD_percent_change_90d,quote_USD_market_cap,quote_USD_market_cap_dominance,quote_USD_fully_diluted_market_cap,quote_USD_tvl,quote_USD_last_updated,platform_id,platform_name,platform_symbol,platform_slug,platform_token_address
0,1,Bitcoin,BTC,bitcoin,12415,2010-07-13T00:00:00.000Z,"[mineable, pow, sha-256, store-of-value, state...",2.100000e+07,1.993279e+07,1.993279e+07,False,NaN,1,NaN,NaN,None,2025-10-12T08:03:00.000Z,111651.945460,8.055841e+10,-56.8512,-0.127119,1.067055,-10.455136,-3.002706,-6.626034,-8.928733,2.225535e+12,59.6446,2.344691e+12,None,2025-10-12T08:03:00.000Z,NaN,NaN,NaN,NaN,NaN
1,1027,Ethereum,ETH,ethereum,10885,2015-08-07T00:00:00.000Z,"[pos, smart-contracts, ethereum-ecosystem, coi...",NaN,1.207033e+08,1.207033e+08,True,NaN,2,NaN,NaN,None,2025-10-12T08:03:00.000Z,3830.591321,4.889625e+10,-57.0916,-0.251806,1.788782,-16.942990,-15.339864,-17.299706,25.941426,4.623648e+11,12.3914,4.623648e+11,None,2025-10-12T08:03:00.000Z,NaN,NaN,NaN,NaN,NaN
2,825,Tether USDt,USDT,tether,151696,2015-02-25T00:00:00.000Z,"[stablecoin, asset-backed-stablecoin, usd-stab...",NaN,1.797955e+11,1.820090e+11,True,NaN,3,NaN,NaN,None,2025-10-12T08:03:00.000Z,1.000999,1.870124e+11,-55.0452,0.011075,0.006779,0.090615,0.101892,0.117048,0.088305,1.799751e+11,4.8234,1.821908e+11,None,2025-10-12T08:03:00.000Z,1027.0,Ethereum,ETH,ethereum,0xdac17f958d2ee523a2206206994597c13d831ec7
3,1839,BNB,BNB,bnb,2803,2017-07-25T00:00:00.000Z,"[marketplace, centralized-exchange, payments, ...",NaN,1.391822e+08,1.391822e+08,False,NaN,4,NaN,NaN,None,2025-10-12T08:04:00.000Z,1159.495623,6.850301e+09,-40.7257,0.239323,6.016738,-1.618365,28.201545,37.120777,64.162759,1.613812e+11,4.3250,1.613812e+11,None,2025-10-12T08:04:00.000Z,NaN,NaN,NaN,NaN,NaN
4,52,XRP,XRP,xrp,1717,2013-08-04T00:00:00.000Z,"[medium-of-exchange, enterprise-solutions, xrp...",1.000000e+11,5.991605e+10,9.998579e+10,False,NaN,5,NaN,NaN,None,2025-10-12T08:03:00.000Z,2.391445,8.436375e+09,-58.2028,-0.060270,-0.900938,-22.050945,-21.999288,-26.130340,-19.051764,1.432859e+11,3.8401,2.391445e+11,None,2025-10-12T08:03:00.000Z,NaN,NaN,NaN,NaN,NaN
5,5426,Solana,SOL,solana,1015,2020-04-10T00:00:00.000Z,"[pos, platform, solana-ecosystem, cms-holdings...",NaN,5.465649e+08,6.117273e+08,True,NaN,6,5.252369e+08,9.591800e+10,None,2025-10-12T08:03:00.000Z,182.618551,1.028091e+10,-49.9449,0.000190,-0.218206,-22.872343,-23.123426,-8.006577,9.307947,9.981289e+10,2.6750,1.117128e+11,None,2025-10-12T08:03:00.000Z,NaN,NaN,NaN,NaN,NaN
6,3408,USDC,USDC,usd-coin,32769,2018-10-08T00:00:00.000Z,"[medium-of-exchange, stablecoin, asset-backed-...",NaN,7.548192e+10,7.548192e+10,False,NaN,7,6.090122e+10,6.089904e+10,None,2025-10-12T08:03:00.000Z,0.999964,2.061262e+10,-60.5008,-0.003580,0.013891,0.026889,0.046208,0.026775,0.012737,7.547922e+10,2.0229,7.547922e+10,None,2025-10-12T08:03:00.000Z,1027.0,Ethereum,ETH,ethereum,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48
7,1958,TRON,TRX,tron,1239,2017-09-13T00:00:00.000Z,"[media, payments, tron-ecosystem, layer-1, dwf...",NaN,9.466722e+10,9.466724e+10,True,NaN,8,9.466789e+10,2.989171e+10,None,2025-10-12T08:03:00.000Z,0.315753,9.404629e+08,-54.6677,-0.030341,-0.359028,-8.056992,-9.396391,-11.284863,4.128547,2.989150e+10,0.8011,2.989150e+10,None,2025-10-12T08:03:00.000Z,NaN,NaN,NaN,NaN,NaN
8,74,Dogecoin,DOGE,dogecoin,1317,2013-12-15T00:00:00.000Z,"[mineable, pow, scrypt, medium-of-exchange, me...",NaN,1.513164e+11,1.513164e+11,True,NaN,9,NaN,NaN,None,2025-10-12T08:04:00.000Z,0.190066,4.416382e+09,-62.0353,-0.339332,-0.615159,-28.051087,-27.006063,-22.705340,-8.366212,2

In [44]:
df_crypto.columns

Index(['id', 'name', 'symbol', 'slug', 'num_market_pairs', 'date_added',
       'tags', 'max_supply', 'circulating_supply', 'total_supply',
       'infinite_supply', 'platform', 'cmc_rank',
       'self_reported_circulating_supply', 'self_reported_market_cap',
       'tvl_ratio', 'last_updated', 'quote_USD_price', 'quote_USD_volume_24h',
       'quote_USD_volume_change_24h', 'quote_USD_percent_change_1h',
       'quote_USD_percent_change_24h', 'quote_USD_percent_change_7d',
       'quote_USD_percent_change_30d', 'quote_USD_percent_change_60d',
       'quote_USD_percent_change_90d', 'quote_USD_market_cap',
       'quote_USD_market_cap_dominance', 'quote_USD_fully_diluted_market_cap',
       'quote_USD_tvl', 'quote_USD_last_updated', 'platform_id',
       'platform_name', 'platform_symbol', 'platform_slug',
       'platform_token_address'],
      dtype='object')

In [45]:
df_status.columns

Index(['timestamp', 'error_code', 'error_message', 'elapsed', 'credit_count',
       'notice', 'total_count'],
      dtype='object')

# PHASE 2: Data Storage & Persistence

## What's Next
Now that we have successfully collected our cryptocurrency data from the API (completed in the cells above), we need to store this data for analysis and future use.

## Current Status
- **API Data Collected**: We have `df_crypto` with 10 cryptocurrency records
- **API Metadata Captured**: We have `df_status` with API call information
- **Data Structure Analyzed**: We know what columns and data types we're working with

---

## Step 1: CSV File Storage (Append Mode)

**What the next Python cell does**: 
The `save_dataframes_to_raw()` function will save our DataFrames to CSV files in append mode.

**Purpose**: 
- Create persistent historical data files
- Each API call adds more records (10 → 20 → 30...)
- Provides offline backup of all collected data

**Files created**:
- `cryptocurrency_data.csv` - Main crypto market data
- `api_status.csv` - API call metadata and performance metrics

---

## Step 2: Database Connection Setup

**What the next Python cell does**: 
The `get_database_connection()` function will test our SQL Server connection.

**Purpose**:
- Verify database connectivity before attempting data insertion
- Provide helpful error messages if database setup is missing
- Use environment variables for secure connection parameters

**Expected outcome**: 
- Success: "Database connection established"
- Failure: Detailed setup instructions

---

## CSV File Storage Implementation

**What this cell does**: Define and execute a function to save DataFrames as CSV files in append mode.
**Result**: Creates/updates persistent CSV files that grow with each API call for historical data tracking.

In [46]:
# 4.1 Save DataFrames to Raw Data Directory (Append Mode)

def save_dataframes_to_raw(df_crypto, df_status):
    """
    Save both cryptocurrency and status DataFrames to persistent raw data files
    Data is APPENDED to existing files - they grow with each API call
    
    Files created:
    - cryptocurrency_data.csv (grows: 10 → 20 → 30 records...)
    - api_status.csv (grows: 1 → 2 → 3 entries...)
    
    Args:
        df_crypto (pd.DataFrame): Cryptocurrency data from API
        df_status (pd.DataFrame): API status/metadata information
    
    Returns:
        dict: Paths to saved files and operation results
    """
    # Define raw data directory (relative path from notebook location)
    raw_data_dir = r'../../01-data/raw'
    
    # Create directory if needed
    os.makedirs(raw_data_dir, exist_ok=True)
    
    # Show absolute path for verification
    abs_path = os.path.abspath(raw_data_dir)
    print(f"TARGET DIRECTORY: {raw_data_dir}")
    print(f"Absolute path: {abs_path}")
    print(f"Directory exists: {os.path.exists(raw_data_dir)}")
    
    # Initialize results
    results = {
        'crypto_path': None,
        'status_path': None,
        'success': False,
        'errors': []
    }
    
    # Define persistent file paths (NO TIMESTAMPS - same files each time)
    crypto_filename = 'cryptocurrency_data.csv'
    status_filename = 'api_status.csv'
    crypto_path = os.path.join(raw_data_dir, crypto_filename)
    status_path = os.path.join(raw_data_dir, status_filename)
    
    # Add collection timestamp to DataFrames for tracking
    current_time = datetime.now()
    
    # SAVE CRYPTOCURRENCY DATA (APPEND MODE)
    if not df_crypto.empty:
        try:
            # Add collection timestamp to crypto data
            df_crypto_with_timestamp = df_crypto.copy()
            df_crypto_with_timestamp['collection_timestamp'] = current_time
            
            # Check if file exists
            crypto_file_exists = os.path.exists(crypto_path)
            
            if crypto_file_exists:
                # APPEND to existing file (no headers)
                df_crypto_with_timestamp.to_csv(crypto_path, mode='a', header=False, index=False)
                action = "APPENDED"
                
                # Read back to get total record count
                existing_df = pd.read_csv(crypto_path)
                total_crypto_records = len(existing_df)
            else:
                # CREATE new file with headers
                df_crypto_with_timestamp.to_csv(crypto_path, index=False)
                action = "CREATED"
                total_crypto_records = len(df_crypto_with_timestamp)
            
            results['crypto_path'] = crypto_path
            
            print(f"CRYPTOCURRENCY DATA - {action}")
            print(f"   • File: {crypto_filename}")
            print(f"   • Location: {crypto_path}")
            print(f"   • This API call: {len(df_crypto)} new records")
            print(f"   • Total records: {total_crypto_records:,} (cumulative)")
            print(f"   • Columns: {len(df_crypto_with_timestamp.columns)} data fields")
            print(f"   • File size: {os.path.getsize(crypto_path):,} bytes")
            
        except Exception as e:
            error_msg = f"Failed to save crypto data: {e}"
            results['errors'].append(error_msg)
            print(f"ERROR: {error_msg}")
    else:
        results['errors'].append("Crypto DataFrame is empty")
        print("WARNING: Crypto DataFrame is empty - no crypto data saved")
    
    # SAVE API STATUS DATA (APPEND MODE)
    if not df_status.empty:
        try:
            # Add collection timestamp to status data
            df_status_with_timestamp = df_status.copy()
            df_status_with_timestamp['collection_timestamp'] = current_time
            
            # Check if file exists
            status_file_exists = os.path.exists(status_path)
            
            if status_file_exists:
                # APPEND to existing file (no headers)
                df_status_with_timestamp.to_csv(status_path, mode='a', header=False, index=False)
                action = "APPENDED"
                
                # Read back to get total record count
                existing_df = pd.read_csv(status_path)
                total_status_records = len(existing_df)
            else:
                # CREATE new file with headers
                df_status_with_timestamp.to_csv(status_path, index=False)
                action = "CREATED"
                total_status_records = len(df_status_with_timestamp)
            
            results['status_path'] = status_path
            
            print(f"\nAPI STATUS DATA - {action}")
            print(f"   • File: {status_filename}")
            print(f"   • Location: {status_path}")
            print(f"   • This API call: {len(df_status)} new entries")
            print(f"   • Total entries: {total_status_records:,} (cumulative)")
            print(f"   • Columns: {len(df_status_with_timestamp.columns)} metadata fields")
            print(f"   • File size: {os.path.getsize(status_path):,} bytes")
            
        except Exception as e:
            error_msg = f"Failed to save status data: {e}"
            results['errors'].append(error_msg)
            print(f"ERROR: {error_msg}")
    else:
        results['errors'].append("Status DataFrame is empty")
        print("WARNING: Status DataFrame is empty - no status data saved")
    
    # Set overall success status
    results['success'] = len(results['errors']) == 0
    
    # Summary
    print(f"\nRAW DATA STORAGE SUMMARY")
    print(f"   • Collection Time: {current_time.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"   • Target Directory: {raw_data_dir}")
    print(f"   • Mode: APPEND (files grow with each API call)")
    print(f"   • Files Updated: {sum([1 for path in [results['crypto_path'], results['status_path']] if path])}")
    print(f"   • Status: {'SUCCESS' if results['success'] else 'PARTIAL'}")
    
    if results['errors']:
        print(f"   • Errors: {len(results['errors'])}")
        for error in results['errors']:
            print(f"     - {error}")
    
    print(f"\nNEXT API CALL WILL:")
    if results['crypto_path']:
        print(f"   • Add {len(df_crypto)} more records to cryptocurrency_data.csv")
    if results['status_path']:
        print(f"   • Add {len(df_status)} more entries to api_status.csv")
    
    return results


save_results = save_dataframes_to_raw(df_crypto, df_status)

TARGET DIRECTORY: ../../01-data/raw
Absolute path: c:\Users\Morobang\Documents\GitHub\CryptoSphere-Analytics-Platform\01-data\raw
Directory exists: True
CRYPTOCURRENCY DATA - APPENDED
   • File: cryptocurrency_data.csv
   • Location: ../../01-data/raw\cryptocurrency_data.csv
   • This API call: 10 new records
   • Total records: 40 (cumulative)
   • Columns: 37 data fields
   • File size: 32,979 bytes

API STATUS DATA - APPENDED
   • File: api_status.csv
   • Location: ../../01-data/raw\api_status.csv
   • This API call: 1 new entries
   • Total entries: 4 (cumulative)
   • Columns: 8 metadata fields
   • File size: 366 bytes

RAW DATA STORAGE SUMMARY
   • Collection Time: 2025-10-12 10:23:18
   • Target Directory: ../../01-data/raw
   • Mode: APPEND (files grow with each API call)
   • Files Updated: 2
   • Status: SUCCESS

NEXT API CALL WILL:
   • Add 10 more records to cryptocurrency_data.csv
   • Add 1 more entries to api_status.csv


## Database Connection Setup

**What this cell does**: Establish and test connection to SQL Server database using environment variables.
**Result**: Returns active connection object or provides detailed setup instructions if database is unavailable.

In [47]:
# 4.2 Database Connection Setup

def get_database_connection():
    """
    Establish connection to SQL Server database
    
    Returns:
        pyodbc.Connection: Database connection or None if failed
    """
    try:
        # Database connection parameters from environment variables
        server = os.getenv('SQL_SERVER')
        database = os.getenv('SQL_DATABASE')
        
        conn_str = (
            'DRIVER={ODBC Driver 17 for SQL Server};'
            f'SERVER={server};'
            f'DATABASE={database};'
            'Trusted_Connection=yes;'
        )
        
        connection = pyodbc.connect(conn_str)
        print("Database connection established")
        return connection
        
    except Exception as e:
        print(f"Database connection failed: {e}")
        print("\nSETUP REQUIRED: Database not available")
        print("="*60)
        print("Before running this notebook, you need to:")
        print("")
        print("1. RUN THE DATABASE SETUP SCRIPT:")
        print("   File: 03-sql-processing/00-schema-setup/01-setup_all_layers.sql")
        print("")
        print("2. WHAT THIS SCRIPT DOES:")
        print("   • Creates cryptosphere_analytics database")
        print("   • Sets up Bronze, Silver, Gold layer schemas") 
        print("")
        print("3. HOW TO RUN:")
        print("   • Open SQL Server Management Studio (SSMS)")
        print("   • Connect to your SQL Server instance")
        print("   • Open and execute the setup script")
        print("   • Verify database 'cryptosphere_analytics' exists")
        print("")
        print("4. AFTER SETUP:")
        print("   • Re-run this notebook cell")
        print("   • Database storage will work automatically")
        print("="*60)
        return None

# Test database connection
print("TESTING DATABASE CONNECTION...")
test_conn = get_database_connection()
if test_conn:
    test_conn.close()
    print("Database ready - proceeding with data storage!")
else:
    print("Database not available - data will be saved to CSV only")
    print("Run the setup script above to enable database storage")

TESTING DATABASE CONNECTION...
Database connection established
Database ready - proceeding with data storage!


## Step 3: Database Data Insertion

**What the previous Python cell does**: 
The `save_to_database()` function inserts our DataFrames into SQL Server using a two-step process.

**Implementation Strategy**: 
1. **Insert API status** → Get batch_id for data lineage tracking
2. **Insert cryptocurrency data** → Link each crypto record to the batch_id

---

In [ ]:
# 4.3 Save to SQL Database (New Two-Table Schema)

def save_to_database(df_crypto, df_status):
    """
    Insert both cryptocurrency and status data into SQL Server Bronze layer 
    using the new two-table schema with proper stored procedures
    
    Args:
        df_crypto (pd.DataFrame): Cryptocurrency data from API
        df_status (pd.DataFrame): API status/metadata information
    
    Returns:
        dict: Results with success counts for both insertions
    """
    if df_crypto.empty and df_status.empty:
        print("No data to insert - both DataFrames are empty")
        return {'crypto_success': 0, 'status_success': 0, 'batch_id': None}
    
    conn = get_database_connection()
    if not conn:
        return {'crypto_success': 0, 'status_success': 0, 'batch_id': None}
    
    results = {
        'crypto_success': 0,
        'status_success': 0,
        'batch_id': None,
        'errors': []
    }
    
    print("INSERTING DATA USING NEW TWO-TABLE SCHEMA...")
    print("Step 1: Insert API status → Step 2: Insert cryptocurrency data")
    
    try:
        cursor = conn.cursor()
        
        # Helper function to handle None/NaN values
        def clean_for_sql(value):
            """Convert pandas NaN/None to SQL NULL, handle different data types including Series"""
            # Handle pandas Series or array-like objects
            if hasattr(value, '__len__') and not isinstance(value, str):
                if len(value) == 1:
                    value = value.iloc[0] if hasattr(value, 'iloc') else value[0]
                elif len(value) > 1:
                    # Convert array/list to JSON string
                    return str(value.tolist()) if hasattr(value, 'tolist') else str(list(value))
            
            # Handle scalar values
            if value is None:
                return None
            if pd.isna(value):
                return None
            if isinstance(value, (int, float)):
                if pd.isna(value):  # Check for NaN in numeric types
                    return None
                return value
            return str(value)
        
        # STEP 1: Insert API Status Data and get batch_id
        if not df_status.empty:
            print(f"\nSTEP 1: Inserting API status ({len(df_status)} entries)...")
            
            for index, status_row in df_status.iterrows():
                try:
                    cursor.execute("""
                        EXEC bronze.sp_insert_api_status
                            @timestamp = ?,
                            @error_code = ?,
                            @error_message = ?,
                            @elapsed = ?,
                            @credit_count = ?,
                            @notice = ?,
                            @total_count = ?,
                            @api_endpoint = ?,
                            @request_parameters = ?,
                            @response_size_bytes = ?,
                            @records_processed = ?
                    """, (
                        clean_for_sql(status_row.get('timestamp')),
                        clean_for_sql(status_row.get('error_code', 0)),
                        clean_for_sql(status_row.get('error_message')),
                        clean_for_sql(status_row.get('elapsed')),
                        clean_for_sql(status_row.get('credit_count')),
                        clean_for_sql(status_row.get('notice')),
                        clean_for_sql(status_row.get('total_count')),
                        'https://pro-api.coinmarketcap.com/v1/cryptocurrency/listings/latest',
                        'start=1&limit=10&convert=USD',
                        None,  # response_size_bytes
                        len(df_crypto) if not df_crypto.empty else 0
                    ))
                    
                    # Get the result with batch_id
                    result = cursor.fetchone()
                    if result and result[2] == 'SUCCESS':  # status column
                        results['status_success'] += 1
                        results['batch_id'] = result[1]  # batch_id column
                        print(f"   API status inserted successfully")
                        print(f"   Batch ID: {results['batch_id']}")
                    else:
                        error_msg = result[3] if result else "Unknown error"
                        results['errors'].append(f"Status insert failed: {error_msg}")
                        print(f"   ❌ Status insert failed: {error_msg}")
                        
                except Exception as e:
                    error_msg = f"Error inserting status: {str(e)[:100]}"
                    results['errors'].append(error_msg)
                    print(f"   ❌ {error_msg}")
                    continue
        
        # STEP 2: Insert Cryptocurrency Data using batch_id
        if not df_crypto.empty and results['batch_id']:
            print(f"\nSTEP 2: Inserting cryptocurrency data ({len(df_crypto)} records)...")
            print(f"   Using batch_id: {results['batch_id']}")
            
            for index, crypto_row in df_crypto.iterrows():
                try:
                    cursor.execute("""
                        EXEC bronze.sp_insert_cryptocurrency_data
                            @batch_id = ?,
                            @id = ?,
                            @name = ?,
                            @symbol = ?,
                            @slug = ?,
                            @num_market_pairs = ?,
                            @date_added = ?,
                            @max_supply = ?,
                            @circulating_supply = ?,
                            @total_supply = ?,
                            @infinite_supply = ?,
                            @platform = ?,
                            @platform_id = ?,
                            @platform_name = ?,
                            @platform_symbol = ?,
                            @platform_slug = ?,
                            @platform_token_address = ?,
                            @cmc_rank = ?,
                            @self_reported_circulating_supply = ?,
                            @self_reported_market_cap = ?,
                            @tvl_ratio = ?,
                            @last_updated = ?,
                            @quote_USD_price = ?,
                            @quote_USD_volume_24h = ?,
                            @quote_USD_volume_change_24h = ?,
                            @quote_USD_percent_change_1h = ?,
                            @quote_USD_percent_change_24h = ?,
                            @quote_USD_percent_change_7d = ?,
                            @quote_USD_percent_change_30d = ?,
                            @quote_USD_percent_change_60d = ?,
                            @quote_USD_percent_change_90d = ?,
                            @quote_USD_market_cap = ?,
                            @quote_USD_market_cap_dominance = ?,
                            @quote_USD_fully_diluted_market_cap = ?,
                            @quote_USD_tvl = ?,
                            @quote_USD_last_updated = ?,
                            @tags = ?
                    """, (
                        results['batch_id'],
                        clean_for_sql(crypto_row.get('id')),
                        clean_for_sql(crypto_row.get('name')),
                        clean_for_sql(crypto_row.get('symbol')),
                        clean_for_sql(crypto_row.get('slug')),
                        clean_for_sql(crypto_row.get('num_market_pairs')),
                        clean_for_sql(crypto_row.get('date_added')),
                        clean_for_sql(crypto_row.get('max_supply')),
                        clean_for_sql(crypto_row.get('circulating_supply')),
                        clean_for_sql(crypto_row.get('total_supply')),
                        clean_for_sql(crypto_row.get('infinite_supply')),
                        clean_for_sql(crypto_row.get('platform')),
                        clean_for_sql(crypto_row.get('platform_id')),
                        clean_for_sql(crypto_row.get('platform_name')),
                        clean_for_sql(crypto_row.get('platform_symbol')),
                        clean_for_sql(crypto_row.get('platform_slug')),
                        clean_for_sql(crypto_row.get('platform_token_address')),
                        clean_for_sql(crypto_row.get('cmc_rank')),
                        clean_for_sql(crypto_row.get('self_reported_circulating_supply')),
                        clean_for_sql(crypto_row.get('self_reported_market_cap')),
                        clean_for_sql(crypto_row.get('tvl_ratio')),
                        clean_for_sql(crypto_row.get('last_updated')),
                        clean_for_sql(crypto_row.get('quote_USD_price')),
                        clean_for_sql(crypto_row.get('quote_USD_volume_24h')),
                        clean_for_sql(crypto_row.get('quote_USD_volume_change_24h')),
                        clean_for_sql(crypto_row.get('quote_USD_percent_change_1h')),
                        clean_for_sql(crypto_row.get('quote_USD_percent_change_24h')),
                        clean_for_sql(crypto_row.get('quote_USD_percent_change_7d')),
                        clean_for_sql(crypto_row.get('quote_USD_percent_change_30d')),
                        clean_for_sql(crypto_row.get('quote_USD_percent_change_60d')),
                        clean_for_sql(crypto_row.get('quote_USD_percent_change_90d')),
                        clean_for_sql(crypto_row.get('quote_USD_market_cap')),
                        clean_for_sql(crypto_row.get('quote_USD_market_cap_dominance')),
                        clean_for_sql(crypto_row.get('quote_USD_fully_diluted_market_cap')),
                        clean_for_sql(crypto_row.get('quote_USD_tvl')),
                        clean_for_sql(crypto_row.get('quote_USD_last_updated')),
                        clean_for_sql(crypto_row.get('tags'))
                    ))
                    
                    # Get stored procedure result
                    result = cursor.fetchone()
                    if result and result[2] == 'SUCCESS':  # status column
                        results['crypto_success'] += 1
                        
                        # Progress indicator every 3 records
                        if results['crypto_success'] % 3 == 0:
                            print(f"   ✅ Progress: {results['crypto_success']}/{len(df_crypto)} records inserted")
                    else:
                        error_msg = result[3] if result else "Unknown error"
                        results['errors'].append(f"Failed to insert {crypto_row.get('symbol', 'Unknown')}: {error_msg}")
                        print(f"   ❌ Failed to insert {crypto_row.get('symbol', 'Unknown')}: {error_msg}")
                        
                except Exception as e:
                    error_msg = f"Error inserting {crypto_row.get('symbol', 'Unknown')}: {str(e)[:100]}"
                    results['errors'].append(error_msg)
                    print(f"   ❌ {error_msg}")
                    continue
        
        # Commit all changes
        conn.commit()
        
        # Final Summary
        print(f"\nDATABASE STORAGE COMPLETE!")
        print(f"   API Status: {results['status_success']}/1 inserted")
        print(f"   Cryptocurrency Data: {results['crypto_success']}/{len(df_crypto)} inserted")
        print(f"   Batch ID: {results['batch_id']}")
        print(f"   Tables: bronze.api_response_status + bronze.cryptocurrency_data")

        if results['crypto_success'] > 0:
            crypto_symbols = ', '.join(df_crypto['symbol'].head(3).tolist())
            print(f"   Sample symbols: {crypto_symbols}")
        
        if results['errors']:
            print(f"   Errors: {len(results['errors'])}")
            for error in results['errors'][:3]:  # Show first 3 errors
                print(f"      - {error}")
        
        success_rate = ((results['status_success'] + results['crypto_success']) / 
                       (len(df_status) + len(df_crypto))) * 100
        print(f"   Overall Success Rate: {success_rate:.1f}%")
        
    except Exception as e:
        print(f"\n❌ Database operation failed: {e}")
        conn.rollback()
        results['errors'].append(f"Database operation failed: {e}")
    finally:
        conn.close()
    
    return results

# Execute database storage using NEW schema
print("SAVING TO DATABASE WITH NEW TWO-TABLE SCHEMA...")
print("Tables: bronze.api_response_status + bronze.cryptocurrency_data")
db_results = save_to_database(df_crypto, df_status)

🔄 SAVING TO DATABASE WITH NEW TWO-TABLE SCHEMA...
Tables: bronze.api_response_status + bronze.cryptocurrency_data
Database connection established
INSERTING DATA USING NEW TWO-TABLE SCHEMA...
Step 1: Insert API status → Step 2: Insert cryptocurrency data

🔄 STEP 1: Inserting API status (1 entries)...
   ✅ API status inserted successfully
   📋 Batch ID: 1CA13853-9A7B-44AA-8180-0905EC7712ED

🔄 STEP 2: Inserting cryptocurrency data (10 records)...
   Using batch_id: 1CA13853-9A7B-44AA-8180-0905EC7712ED
   ✅ Progress: 3/10 records inserted
   ✅ Progress: 6/10 records inserted
   ✅ Progress: 9/10 records inserted

🎯 DATABASE STORAGE COMPLETE!
   📊 API Status: 1/1 inserted
   🪙 Cryptocurrency Data: 10/10 inserted
   🔗 Batch ID: 1CA13853-9A7B-44AA-8180-0905EC7712ED
   📍 Tables: bronze.api_response_status + bronze.cryptocurrency_data
   🚀 Sample symbols: BTC, ETH, USDT
   📈 Overall Success Rate: 100.0%
